In [ ]:
%load_ext autoreload
%autoreload 2

import os
import torchio as tio

# pra usar cpu, descomentar linha abaixo
#os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
import nibabel as nib

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, CSVLogger, EarlyStopping
from tensorflow.keras.layers import Input, Conv3D, MaxPooling3D, Flatten, Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2
from tensorflow.keras import layers, models, Input, Model

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import gc
import seaborn as sns

from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from PIL import Image
import tempfile
from math import ceil
import random
from tensorflow.keras import backend as K

#import wandb

import utils.processamento_dados as proc_dados
import utils.metricas_e_visualizacao as met_vil

In [ ]:
from tensorflow.keras import mixed_precision

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)  # Evita uso excessivo de memória
        print("GPU habilitada com sucesso!")
        print("Memory Growth habilitado para a GPU")
    except RuntimeError as e:
        print(e)

mixed_precision.set_global_policy("mixed_float16")

tf.get_logger().setLevel('ERROR')

In [ ]:
# FUNÇÕES

def create_model_3d(input_shape, n_classes):
    inputs = Input(shape=input_shape)  # (D, H, W, C)

    # Camada 1
    x = layers.Conv3D(4, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.3)(x)
    x = layers.AveragePooling3D(pool_size=(3, 3, 3), padding='same')(x)
    x = layers.Dropout(0.3)(x)

    # Camada 2
    x = layers.Conv3D(8, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.3)(x)
    x = layers.AveragePooling3D(pool_size=(3, 3, 3), padding='same')(x)
    x = layers.Dropout(0.3)(x)

    # Camada 3
    x = layers.Conv3D(16, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.3)(x)
    x = layers.AveragePooling3D(pool_size=(3, 3, 3), padding='same')(x)
    x = layers.Dropout(0.3)(x)

    # # Camada 4
    # x = layers.Conv3D(32, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(x)
    # x = layers.BatchNormalization()(x)
    # x = layers.LeakyReLU(negative_slope=0.3)(x)
    # # x = layers.MaxPooling3D(pool_size=(2, 2, 2), padding='same')(x)
    # x = layers.Dropout(0.3)(x)

    # Flatten e densas
    x = layers.Flatten()(x)

    x = layers.Dense(16, kernel_regularizer=l2(0.01))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.LeakyReLU(negative_slope=0.3)(x)

    outputs = layers.Dense(n_classes, activation='softmax')(x)

    model = models.Model(inputs=inputs, outputs=outputs)

    return model

In [ ]:
from tensorflow.keras import mixed_precision

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)  # Evita uso excessivo de memória
        print("GPU habilitada com sucesso!")
        print("Memory Growth habilitado para a GPU")
    except RuntimeError as e:
        print(e)

mixed_precision.set_global_policy("mixed_float16")

tf.get_logger().setLevel('ERROR')

# Optional deterministic settings from the OASIS-only notebook
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ['TF_CUDNN_USE_AUTOTUNE'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [ ]:
# transform_name = None

# for item in transformations:
#     results_dir = f"{results_dir_base}/test_{item}"
#     if (not os.path.exists(results_dir)) or ((os.path.exists(results_dir) and len(os.listdir(results_dir)) < 3)):
#         os.makedirs(results_dir, exist_ok=True)
#         print(f"pasta test_{item} criada")
#         transform_name = item
#         break

In [ ]:
# Configuration block: choose the dataset and problem variant you want to run
# mode = 'adni_multiclass'  -> ADNI data, multiclass categorical model
# mode = 'adni_binary'      -> ADNI data, binary model
# mode = 'oasis_binary'     -> OASIS data, binary model
mode = 'oasis_binary'

# Definindo caminhos
dir_base = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn"

if mode == 'oasis_binary':
    train_dir = f'{dir_base}/OASIS/OASIS_1_FSL_NORMALIZED/train'
    val_dir = f'{dir_base}/OASIS/OASIS_1_FSL_NORMALIZED/validation'
    results_dir_base = f'{dir_base}/OASIS/OASIS_NORMALIZED_MR2/results/ruidos_binary'
    class_names = ['0.0', '0.5', '1.0']
    binary_classes = [0, 1]
    use_binary = True
    batch_size = 10
    epochs = 100
    augment_train = True
    augment_val = False
    split_val_test = True
    use_pdf_reports = False
elif mode == 'adni_binary':
    train_dir = f'{dir_base}/ADNI/ADNI_NORMALIZED/train'
    val_dir = f'{dir_base}/ADNI/ADNI_NORMALIZED/validation'
    results_dir_base = f'{dir_base}/ADNI/ADNI_NORMALIZED/results/ruidos_emci_lmci'
    class_names = ['cn', 'mci', 'ad']
    binary_classes = [0, 1]
    use_binary = True
    batch_size = 64
    epochs = 200
    augment_train = True
    augment_val = True
    split_val_test = False
    use_pdf_reports = True
else:
    train_dir = f'{dir_base}/ADNI/ADNI_NORMALIZED/train'
    val_dir = f'{dir_base}/ADNI/ADNI_NORMALIZED/validation'
    results_dir_base = f'{dir_base}/ADNI/ADNI_NORMALIZED/results/ruidos_emci_lmci'
    class_names = ['emci', 'lmci']
    binary_classes = None
    use_binary = False
    batch_size = 64
    epochs = 200
    augment_train = True
    augment_val = True
    split_val_test = False
    use_pdf_reports = True

os.makedirs(results_dir_base, exist_ok=True)
results_dir = results_dir_base

transformations = ['noise']

In [ ]:
# Optional PDF report generation for the selected mode
if use_pdf_reports:
    pdf_reports_dir = os.path.join(results_dir, "relatorios_10_pessoas")
    met_vil.generate_axial_pdf_reports_no_prediction(
        images=train_images,
        true_labels_onehot=train_labels,
        class_names=class_names,
        output_dir=pdf_reports_dir,
        max_samples=100
    )

In [ ]:
# Load validation data after configuration
if mode == 'oasis_binary':
    val_images, val_labels, val_paths, _ = proc_dados.load_nifti_data_balanced_preallocated(
        val_dir,
        class_names,
        augment=augment_val
    )
    print(f"N validation: {len(val_paths)}")

    if split_val_test:
        val_images, test_images, val_labels, test_labels, val_paths, test_paths = train_test_split(
            val_images,
            val_labels,
            val_paths,
            train_size=0.6,
            test_size=0.4,
            random_state=42
        )
        print(f"N validation depois: {len(val_paths)}")
        print(f"N teste: {len(test_paths)}")
else:
    val_images, val_labels, val_paths, _ = proc_dados.load_nifti_data_balanced_preallocated(
        val_dir,
        class_names,
        augment=augment_val
    )
    print(f"N validation: {len(val_paths)}")

In [ ]:
# Build the model based on the selected mode
n_classes = len(class_names) if not use_binary else 2
model = create_model_3d(train_images[2].shape, n_classes)

if use_binary:
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['binary_accuracy'])
else:
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005), loss='categorical_crossentropy', metrics=['categorical_accuracy'])

model.summary()

In [ ]:
steps_per_epoch = len(train_paths) // batch_size
validation_steps = len(val_paths) // batch_size

train_generator = proc_dados.nifti_data_generator_3d(train_images, train_labels, batch_size)
val_generator = proc_dados.nifti_data_generator_3d(val_images, val_labels, batch_size)

new_model_name_ker = f"noise_model_{mode}_{epochs}_epochs_batch_{batch_size}_{n_classes}_classes.keras"

# Parar caso fique {patience} épocas sem melhora
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=30,
    verbose=1
)

# Defina o nome do arquivo para salvar o melhor modelo
if use_binary:
    monitor_metric = 'val_binary_accuracy'
else:
    monitor_metric = 'val_categorical_accuracy'

model_checkpoint_callback = ModelCheckpoint(
    filepath=os.path.join(results_dir, new_model_name_ker),
    monitor=monitor_metric,
    save_best_only=True,
    mode='max',
)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, verbose=1)

log_path = os.path.join(results_dir, 'log_treino_ruido.csv')
csv_log = CSVLogger(log_path, append=False)

In [ ]:
print(f"Iniciando treinamento do modelo {new_model_name_ker} para classes {class_names}")

# Treinamento
history = model.fit(
    train_generator,
    epochs=epochs,
    verbose=1,
    validation_data=val_generator,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=[model_checkpoint_callback, reduce_lr, csv_log, early_stopping]
)

In [ ]:
# Plotando o histórico de treinamento após o treinamento
met_vil.plot_training_history(history, f"{results_dir}/final")

### PREDIÇÃO TREINO

In [ ]:
# Optional PDF report generation for train set predictions
if use_pdf_reports:
    pdf_reports_dir = os.path.join(results_dir, "relatorios_axiais_treino")
    met_vil.generate_axial_pdf_reports(
        model=model,
        images=train_images,
        true_labels_onehot=train_labels,
        class_names=class_names,
        output_dir=pdf_reports_dir,
        max_samples=2648
    )

### PREDIÇÃO VALIDAÇÃO

In [ ]:
# Realizar predições para dados do conjunto validação
val_pred_labels, val_true_labels, val_pred = met_vil.get_predictions(val_images, val_labels, batch_size, model)

# Obter métricas da validação e salvá-las em um arquivo
met_vil.get_classification_report(val_true_labels, val_pred_labels, results_dir, 'validation')

### PREDIÇÃO TESTE

In [ ]:
test_images, test_labels, test_paths, _ = proc_dados.load_nifti_data_balanced_preallocated(test_dir, adni_class_names)

# Realizar predições para dados do conjunto validação
test_pred_labels, test_true_labels, test_pred = met_vil.get_predictions(test_images, test_labels, batch_size, model)

# Obter métricas da valiadação e salvá-las em um arquivo
met_vil.get_classification_report(test_true_labels, test_pred_labels, results_dir, 'test_adni')

# Obter matriz de confusão
met_vil.plot_confusion_matrix(test_true_labels, test_pred_labels, results_dir, 'test_adni', adni_class_names)

# Criar pdf com predições
test_pdf_path = os.path.join(results_dir, "test_adni_predictions.pdf")
met_vil.create_pdf(test_paths, test_images, test_true_labels, test_pred_labels, test_pred, test_pdf_path, adni_class_names)

### PREDIÇÃO OASIS

In [ ]:
oasis_images, oasis_labels, oasis_paths, _ = proc_dados.load_nifti_data_balanced_preallocated(oasis_dir, oasis_class_names)

# Realizar predições para dados do conjunto validação
oasis_pred_labels, oasis_true_labels, oasis_pred = met_vil.get_predictions(oasis_images, oasis_labels, batch_size, model)

# Obter métricas da valiadação e salvá-las em um arquivo
met_vil.get_classification_report(oasis_true_labels, oasis_pred_labels, results_dir, 'test_oasis')

# Obter matriz de confusão
cm_adjusted = confusion_matrix(oasis_true_labels, oasis_pred_labels)[0:3, :]
met_vil.plot_custom_confusion_matrix(cm_adjusted, oasis_class_names, adni_class_names, results_dir, 'test_3x5_oasis')

gathered_oasis_pred = []

for i in range(len(oasis_pred_labels)):
    if oasis_pred_labels[i] == 0:
        gathered_oasis_pred.append(0)
    elif oasis_pred_labels[i] < len(adni_class_names)-1 and oasis_pred_labels[i] > 0:
        gathered_oasis_pred.append(1)
    elif oasis_pred_labels[i] == len(adni_class_names)-1:
        gathered_oasis_pred.append(2)

# all_possible_numeric_labels = [0.0, 0.5, 1.0, 2.0, 3.0]

cm_3x3_adjusted = confusion_matrix(oasis_true_labels, gathered_oasis_pred)[0:3, 0:3]

# Chame a nova função para plotar a matriz ajustada (3x5)
met_vil.plot_custom_confusion_matrix(cm_3x3_adjusted, oasis_class_names, oasis_class_names, results_dir, 'test_3x3_oasis')

# Criar pdf com predições
oasis_pdf_path = os.path.join(results_dir, "test_oasis_predictions.pdf")
met_vil.create_pdf(oasis_paths, oasis_images, oasis_true_labels, gathered_oasis_pred, oasis_pred, oasis_pdf_path, oasis_class_names)